# Dataset Preparation Notebook Summary - HUGGINGFACE VER

This notebook prepares the **WIN32 subset** of the **EMBER2024** dataset from Hugging Face for downstream machine learning experiments. It creates four Parquet files that separate the data into malware detection and malware behavior prediction tasks.

> **Note:** This notebook downloads approximately **74 GB** of WIN32 data from the EMBER2024 dataset on Hugging Face. Make sure you have at least **80–100 GB of available disk space** before running the notebook to accommodate the downloaded data and generated Parquet files.

## Data Source

Dataset:
- https://huggingface.co/datasets/joyce8/EMBER2024

Only the **WIN32** samples are used.

## Dataset Sampling

To reduce storage and preprocessing time while maintaining reproducibility:

- A **deterministic random 20% sample** of the WIN32 **training** data is selected.
- The **entire WIN32 test set** is retained.
- Using a fixed random seed ensures the same subset can be regenerated.

This produces:

- Training set (20% deterministic sample)
- Test set (100%)

## Generated Datasets

The notebook creates four Parquet files:

### Malware Detection

Used for binary malware classification.

Training:
- `detection_train.parquet`

Testing:
- `detection_test.parquet`

Each row contains:

| Column | Description |
|---------|-------------|
| `sha256` | Sample SHA-256 hash |
| `label` | Malware label (0 = benign, 1 = malware) |
| `general` | JSON-serialized PE metadata |
| `strings` | JSON-serialized extracted strings/features |
| `imports` | JSON-serialized imported functions/libraries |

Nested dictionaries are serialized into JSON strings before being written to Parquet to ensure a consistent schema.

---

### Malware Behavior Identification

Used for malware family and behavior prediction.

Training:
- `behavior_train.parquet`

Testing:
- `behavior_test.parquet`

Each row contains:

| Column | Description |
|---------|-------------|
| `sha256` | Sample SHA-256 hash |
| `label` | Constant value `1` (only malware samples are included) |
| `family` | Malware family name |
| `behavior` | Reported malware behaviors |
| `mbc` | JSON-serialized Malware Behavior Catalog (MBC) techniques |
| `ttps` | JSON-serialized ATT&CK TTP identifiers |

The behavior dataset only contains malware samples, so the label is fixed to `1`.

## Output

Running the notebook produces four Parquet files:

```
win32_detection_train_20pct.parquet
win32_behavior_train_20pct.parquet
win32_test_detection.parquet
win32_test_behavior.parquet
```

These files provide reproducible, cleaned datasets that can be loaded directly into downstream machine learning pipelines without requiring additional preprocessing of the original Hugging Face dataset.

In [4]:
# install pacakges
!pip install -q huggingface_hub pyarrow tqdm

In [2]:
# imports
from huggingface_hub import hf_hub_download

from pathlib import Path
import zipfile
import json
import hashlib

import pyarrow as pa
import pyarrow.parquet as pq

from tqdm import tqdm


/Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# data paths
# Find the repository root (parent of notebooks/)
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
DATA_DIR.mkdir(exist_ok=True)

TRAIN_DIR = DATA_DIR / "Win32_train"
TEST_DIR = DATA_DIR / "Win32_test"

TRAIN_DIR.mkdir(exist_ok=True)
TEST_DIR.mkdir(exist_ok=True)


DETECTION_PARQUET = DATA_DIR / "win32_detection_train_20pct.parquet"
BEHAVIOR_PARQUET = DATA_DIR / "win32_behavior_train_20pct.parquet"
DETECTION_TEST_PARQUET = DATA_DIR / "win32_test_detection.parquet"
BEHAVIOR_TEST_PARQUET = DATA_DIR / "win32_test_behavior.parquet"

In [ ]:
# download zip files
train_zip = hf_hub_download(
    repo_id="joyce8/EMBER2024",
    repo_type="dataset",
    filename="Win32_train.zip"
)
test_zip = hf_hub_download(
    repo_id="joyce8/EMBER2024",
    repo_type="dataset",
    filename="Win32_test.zip"
)
print(f"Train: {train_zip}")
print(f"Test: {test_zip}")

In [ ]:
# extract zip files
with zipfile.ZipFile(train_zip, "r") as z:
    z.extractall(TRAIN_DIR)

with zipfile.ZipFile(test_zip, "r") as z:
    z.extractall(TEST_DIR)

print("Train files:", len(list(TRAIN_DIR.glob("*.jsonl"))))
print("Test files:", len(list(TEST_DIR.glob("*.jsonl"))))

Train files: 0
Test files: 0


In [14]:
# jsonl reader
def json_stream(folder):
    for file in folder.glob("*.jsonl"):
        with open(file, "r") as f:
            for line in f:
                yield json.loads(line)

In [ ]:
# function to get WIN32 subset
SEED = 42
def keep_sample(row, percent=0.20):
    # hash sha256 ID
    h = hashlib.sha256(
        (row["sha256"] + str(SEED)).encode()
    ).hexdigest()

    normalized = int(h[:8], 16) / 0xffffffff

    return normalized < percent

# row cleaning
def clean_detection_row(row):
    return {
        "sha256": row.get("sha256"),
        "label": row.get("label"),

        # convert nested structures to stable strings
        "general": json.dumps(row.get("general", {})),
        "strings": json.dumps(row.get("strings", {})),
        "imports": json.dumps(row.get("imports", {}))
    }

def clean_behavior_row(row):
    return {
        "sha256": row.get("sha256"),
        "label": 1,
        "family": row.get("family") or "",
        "behavior": row.get("behavior") or [""],
        "mbc": json.dumps(row.get("mbc") or []),
        "ttps": json.dumps(row.get("ttps") or [])
    }

In [45]:
# for malware classification
def save_parquet(input_dir, output_file, cleaner):
    writer = None
    count = 0 # number of saved samples
    seen = set()

    stream = json_stream(input_dir)
    for row in tqdm(stream, desc="Processing Samples"):
        # skip non-WIN32 subset samples
        if not keep_sample(row):
            continue

        cleaned = cleaner(row)
        sha = cleaned["sha256"]
        if sha in seen:
            continue
        seen.add(sha)
        
        table = pa.Table.from_pylist([cleaned])
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )
        # append sample to parquet file
        writer.write_table(table)
        count += 1

    if writer:
        writer.close()
    print(f"Saved {count:,} detection samples")

In [ ]:
# save files to parquet
save_parquet(
    TRAIN_DIR,
    DETECTION_PARQUET,
    clean_detection_row
)

Processing Samples: 3120000it [06:37, 7840.55it/s] 


Saved 312,125 detection samples


In [61]:
# for behavior identification
def save_behavior_parquet(input_dir, output_file, cleaner):
    writer = None
    count = 0 # number of saved samples
    seen = set()

    stream = json_stream(input_dir)
    for row in tqdm(stream, desc="Processing Samples"):
        # skip non-WIN32 subset samples
        if not keep_sample(row):
            continue
        if row.get("label") != 1:
            continue

        cleaned = cleaner(row)
        sha = cleaned["sha256"]
        if sha in seen:
            continue
        seen.add(sha)
        
        table = pa.Table.from_pylist([cleaned])
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )
        # append sample to parquet file
        writer.write_table(table)
        count += 1

    if writer:
        writer.close()
    print(f"Saved {count:,} detection samples")

In [62]:
save_behavior_parquet(
    TRAIN_DIR,
    BEHAVIOR_PARQUET,
    clean_behavior_row
)

Processing Samples: 3120000it [05:56, 8740.41it/s] 


Saved 156,357 detection samples


In [ ]:
# for behavior identification test
def save_behavior_test_parquet(input_dir, output_file, cleaner):
    writer = None
    count = 0 # number of saved samples
    seen = set()

    stream = json_stream(input_dir)
    for row in tqdm(stream, desc="Processing Samples"):
        if row.get("label") != 1:
            continue

        cleaned = cleaner(row)
        sha = cleaned["sha256"]
        if sha in seen:
            continue
        seen.add(sha)
        
        table = pa.Table.from_pylist([cleaned])
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )
        # append sample to parquet file
        writer.write_table(table)
        count += 1

    if writer:
        writer.close()
    print(f"Saved {count:,} detection samples")

# for malware detection test
def save_detection_test_parquet(input_dir, output_file, cleaner):
    writer = None
    count = 0 # number of saved samples
    seen = set()

    stream = json_stream(input_dir)
    for row in tqdm(stream, desc="Processing Samples"):
        cleaned = cleaner(row)
        sha = cleaned["sha256"]
        if sha in seen:
            continue
        seen.add(sha)
        
        table = pa.Table.from_pylist([cleaned])
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )
        # append sample to parquet file
        writer.write_table(table)
        count += 1

    if writer:
        writer.close()
    print(f"Saved {count:,} detection samples")

In [70]:
# test parquet files
save_detection_test_parquet(
    TEST_DIR,
    DETECTION_TEST_PARQUET,
    clean_detection_row
)
save_behavior_test_parquet(
    TEST_DIR,
    BEHAVIOR_TEST_PARQUET,
    clean_behavior_row
)

Processing Samples: 720000it [02:30, 4795.87it/s]


Saved 359,994 detection samples


Processing Samples: 720000it [01:45, 6801.57it/s] 


Saved 180,000 detection samples
